# VAE 简明总结

这个 notebook 用一个尽量简单的版本总结当前 VAE 阶段的重点，帮助你把代码、训练目标和输出结果串起来。

## 1. 我们现在在做什么

当前目标不是追求最强生成效果，而是先把 CIFAR-10 上的 VAE 训练闭环跑通。

VAE 的核心流程是：
1. 编码器把输入图片压缩成潜变量分布的参数 $\mu$ 和 $\log\sigma^2$
2. 用重参数化技巧从这个分布里采样出 $z$
3. 解码器根据 $z$ 重建图像
4. 用重构损失和 KL 散度一起训练

## 2. 模型结构直觉

当前模型是一个小型卷积 VAE：
- 输入：`3 x 32 x 32` 的 CIFAR-10 彩色图像
- 编码器：4 层卷积，把图像压缩到一个小特征图
- 分布头：输出 $\mu$ 和 $\log\sigma^2$
- 重参数化：$z = \mu + \sigma \odot \epsilon$
- 解码器：转置卷积逐步还原图像

可以把它理解成：

输入图片 $x$ -> 编码器 -> $(\mu, \log\sigma^2)$ -> 采样 $z$ -> 解码器 -> 重构图 $\hat{x}$

## 3. 为什么 loss 分成两部分

训练目标是：

$$
athcal{L} = athcal{L}_{recon} + eta athcal{L}_{KL}
$$

其中：
- `reconstruction loss`：让重建图像尽量接近原图
- `KL loss`：让潜变量分布不要偏离标准正态太远

一个常见现象是：
- 重构损失下降
- KL 散度上升

这通常不是坏事，而是说明模型开始真的用 latent 空间来携带信息，而不是一直输出接近先验的分布。

## 4. 训练过程在做什么

完整训练并不是一口气把 100 个 epoch 跑完，而是不断重复下面这个小循环：

1. 从 CIFAR-10 的 dataloader 里取出一个 batch 图像
2. 把图像送进编码器，得到 $\mu$ 和 $\log\sigma^2$
3. 用重参数化技巧采样出 latent 向量 $z$
4. 用解码器生成重构图像
5. 计算 total loss、reconstruction loss 和 KL loss
6. 反向传播并更新参数
7. 一个 epoch 结束后记录 loss 曲线，并保存重构图、随机采样图和 checkpoint

也就是说，训练脚本本质上在做两件事：
- 在 batch 级别不断更新模型参数
- 在 epoch 级别不断保存可观察结果

In [ ]:
from pathlib import Path

import torch

from dataset import build_dataloaders
from loss import vae_loss
from model import ConvVAE
from utils import resolve_device

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

device = resolve_device('auto')
train_loader, _ = build_dataloaders(
    data_dir=project_dir / 'data',
    batch_size=16,
    num_workers=2,
)

model = ConvVAE(latent_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, _ = next(iter(train_loader))
images = images.to(device, non_blocking=device.type == 'cuda')

reconstructions, mu, logvar = model(images)
total_loss, recon_loss, kl_loss = vae_loss(reconstructions, images, mu, logvar, beta=1.0)

optimizer.zero_grad(set_to_none=True)
total_loss.backward()
optimizer.step()

print('device =', device)
print('input shape =', tuple(images.shape))
print('reconstruction shape =', tuple(reconstructions.shape))
print('mu shape =', tuple(mu.shape))
print('logvar shape =', tuple(logvar.shape))
print('total loss =', round(total_loss.item(), 4))
print('reconstruction loss =', round(recon_loss.item(), 4))
print('kl loss =', round(kl_loss.item(), 4))

这一个 cell 演示的是“单个 batch 的训练步骤”。真正的 [train.py](VAE/train.py) 会把这个过程放进 epoch 循环里，持续重复很多次。

完整训练时，每个 epoch 还会额外做三件事：
1. 把 total loss、reconstruction loss 和 KL loss 追加到历史记录里
2. 更新 loss 曲线图
3. 保存当前 epoch 的重构图、随机采样图和 checkpoint

In [ ]:
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

outputs_dir = project_dir / 'outputs'
plots_dir = outputs_dir / 'plots'
recon_dir = outputs_dir / 'reconstructions'
sample_dir = outputs_dir / 'samples'

print('project_dir =', project_dir)
print('plots_dir =', plots_dir)
print('recon_dir =', recon_dir)
print('sample_dir =', sample_dir)

In [ ]:
from pathlib import Path
import json

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

plots_dir = project_dir / 'outputs' / 'plots'
history_path = plots_dir / 'loss_history.json'
with history_path.open('r', encoding='utf-8') as f:
    history = json.load(f)

print('已记录 epoch 数:', len(history['total_loss']))
print('最后一个 epoch 的 total loss:', round(history['total_loss'][-1], 4))
print('最后一个 epoch 的 recon loss:', round(history['recon_loss'][-1], 4))
print('最后一个 epoch 的 kl loss:', round(history['kl_loss'][-1], 4))

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

history_path = project_dir / 'outputs' / 'plots' / 'loss_history.json'
with history_path.open('r', encoding='utf-8') as f:
    history = json.load(f)

epochs = range(1, len(history['total_loss']) + 1)
plt.figure(figsize=(10, 6))
plt.plot(epochs, history['total_loss'], label='total loss', linewidth=2)
plt.plot(epochs, history['recon_loss'], label='reconstruction loss', linewidth=2)
plt.plot(epochs, history['kl_loss'], label='kl loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss Curves')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. 查看最终输出结果

下面两格会直接显示最后一个 epoch 的重构图和随机采样图。

你可以重点观察：
1. 重构图是否保留了原图的大致颜色和结构
2. 随机采样图是否已经不是纯噪声
3. 样本是否还比较模糊，这在当前这个简化版 VAE 里是正常的

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

recon_path = project_dir / 'outputs' / 'reconstructions' / 'epoch_100.png'
image = mpimg.imread(recon_path)
plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis('off')
plt.title('Reconstructions at Epoch 100')
plt.show()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

project_dir = Path.cwd()
if project_dir.name != 'VAE':
    project_dir = project_dir / 'VAE'

sample_path = project_dir / 'outputs' / 'samples' / 'epoch_100.png'
image = mpimg.imread(sample_path)
plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis('off')
plt.title('Random Samples at Epoch 100')
plt.show()

## 6. 这份实现里最值得记住的三件事

1. VAE 不是直接学一个固定向量，而是学一个分布。
2. 重参数化技巧让采样这一步也能参与反向传播。
3. 好的 VAE 训练通常不是让 KL 一直很小，而是让重构和 KL 达到一个合理平衡。

## 7. 下一步可以做什么

如果你继续往下学，最自然的三个方向是：
1. 调 `beta`，观察 KL 和重构之间的平衡变化
2. 调 `latent_dim`，观察 latent 空间容量变化
3. 进入 CVAE，让模型学会按类别条件生成